In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
import ot
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from networkx.algorithms import bipartite
import glob 
from functions import *
from skbio.stats.distance import DistanceMatrix, permanova, mantel, anosim
from scipy.spatial.distance import pdist, squareform

In [ ]:
folder = r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol_pollinators"
path_files = glob.glob(os.path.join(folder, "*.csv"))

In [ ]:

def create_adjacency_matrix(file):
    # Carica il dataframe
    df = pd.read_csv(file)
    df_dati = df.iloc[:, 1:]
    
    # togliamo se ci sono caratteri strani, li sostituiamo con nan e poi con 0
    df_dati = df_dati.apply(pd.to_numeric, errors='coerce')
    df_dati = df_dati.fillna(0)
    
    # Estrai la matrice di incidenza B (rimuovendo la prima colonna dei nomi)
    B = df_dati.iloc[:, 1:].to_numpy()
    n_plants, m_pollinators = B.shape
    
    # Creiamo l'effettiva matrice di adiacenza
    total_size = n_plants + m_pollinators
    adj_matrix = np.zeros((total_size, total_size))
  
    adj_matrix[:n_plants, n_plants:] = B
    
    adj_matrix[n_plants:, :n_plants] = B.T
    
    return adj_matrix

# Utilizzo
adj = create_adjacency_matrix(r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol_pollinators/M_PL_070.csv")

print(adj)

In [ ]:
def get_bip_data(file):
    df = pd.read_csv(file)
    
    piante = df.iloc[:, 0].tolist()
    pollinatori = df.columns[1:].tolist()
    
    return piante, pollinatori


In [ ]:
metadata = pd.read_csv(r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol_pollinators/references.csv")

In [ ]:
island_mapping = {
    # Continental Island 
    'Amami-Ohsima Island, Japan': 'continental island',
    'Arima Valley': 'continental island',
    "Arthur's Pass, New Zealand": 'continental island',
    'Ashu, Kyoto, Japan': 'continental island',
    'Bristol, England': 'continental island',
    'Cass, New Zealand': 'continental island',
    'Chiloe, Chile': 'continental island',
    'Craigieburn, New Zealand': 'continental island',
    'Hazen Camp, Ellesmere Island, Canada': 'continental island',
    'Hickling, Norfolk, UK': 'continental island',
    'Kibune, Kyoto, Japan': 'continental island',
    'Kyoto City, Japan': 'continental island',
    'Matamata': 'continental island',
    'Melville Island, Canada': 'continental island',
    'Morne Seychellois National Park, Mahé': 'continental island',
    'Mt. Kushigata, Yamanashi Pref., Japan': 'continental island',
    'Mt. Yufu, Japan': 'continental island',
    'Nakaikemi marsh, Fukui Prefecture, Japan': 'continental island',
    'Shelfanger, Norfolk, UK': 'continental island',
    'Tundra, Greenladn': 'continental island',
    'Uummannaq Island, Greenland': 'continental island',
    'Zackenberg': 'continental island',
    
    # Oceanic Island 
    'Black River Gorges National Park, Mauritius': 'oceanic island',
    'Flores, Açores': 'oceanic island',
    'Galapagos': 'oceanic island',
    'Garajonay, Gomera, Spain': 'oceanic island',
    'Mauritius Island': 'oceanic island',
    'Morant Point, Jamaica': 'oceanic island',
    'Puerto Villamil, Isabela Island, Galapagos': 'oceanic island',
    'Syndicate, Dominica': 'oceanic island',
    'Tenerife, Canary Islands': 'oceanic island',
    'Windsor, The Cockpit Country, Jamaica': 'oceanic island'
}

In [ ]:
metadata['island_type'] = metadata['Locality of Study'].map(island_mapping).fillna('mainland')

Per ogni file creiamo un dizionario in cui inseriamo gli attributi di ogni nodo

In [ ]:
all_g = []

for file in path_files:
    pl, po = get_bip_data(file)
    gg = create_adjacency_matrix(file)
    
    nomi_nodi = pl + po 
    attributi_nodi = {}

    for indice, nome in enumerate(nomi_nodi):
        tipo_nodo = "Pianta" if indice < len(pl) else "Impollinatore"
        
        # mettiamo di che genere è: lo facciamo prendendo la prima parola di ogni nome completo
        genere = nome.split()[0] 
        
        attributi_nodi[indice] = {
            "nome": nome,
            "tipo": tipo_nodo,
            "genere": genere  
        }
    island_types = metadata['island_type'].values
    i=island_types[path_files.index(file)-1]
    all_g.append({
        "matrice": gg,
        "attributi": attributi_nodi,
        "island_type": i
    })


In [ ]:
def crea_partizione_genere(attributi_nodi):
    """
    Prende in input il dizionario degli attributi dei nodi e restituisce 
    una lista di liste, dove ogni sottolista contiene gli indici dei nodi 
    appartenenti allo stesso genere.
    """
    raggruppamenti = {}
    
    for indice, dati_nodo in attributi_nodi.items():
        genere = dati_nodo["genere"]
        
        if genere not in raggruppamenti:
            raggruppamenti[genere] = []
            
        # Corretto: aggiungiamo l'indice (il numero) e non l'intero dizionario
        raggruppamenti[genere].append(indice)
        
    return list(raggruppamenti.values())

def crea_dizionario_nomi_genere(attributi_nodi):
    """
    Prende in input il dizionario degli attributi dei nodi e restituisce 
    un dizionario dove la chiave è il genere e il valore è la lista dei nomi.
    """
    dizionario_nomi = {}
    
    for indice, dati_nodo in attributi_nodi.items():
        genere = dati_nodo["genere"]
        nome = dati_nodo["nome"]
        
        if genere not in dizionario_nomi:
            dizionario_nomi[genere] = []
            
        # Aggiungiamo il nome per esteso alla lista di quel genere
        dizionario_nomi[genere].append(nome)
        
    return dizionario_nomi


# --- Integrazione nel tuo ciclo esistente ---

all_g = []

for file in path_files:
    pl, po = get_bip_data(file)
    gg = create_adjacency_matrix(file)
    
    nomi_nodi = pl + po 
    attributi_nodi = {}

    for indice, nome in enumerate(nomi_nodi):
        tipo_nodo = "Pianta" if indice < len(pl) else "Impollinatore"
        
        # Estraiamo il genere (prima parola)
        genere = nome.split()[0] 
        
        attributi_nodi[indice] = {
            "nome": nome,
            "tipo": tipo_nodo,
            "genere": genere  
        }
    
    # 1. Creiamo la lista di liste con gli indici
    partizione_indici = crea_partizione_genere(attributi_nodi)
    
    # 2. Creiamo il dizionario {genere: [nomi]}
    dizionario_nomi = crea_dizionario_nomi_genere(attributi_nodi)
    
    island_types = metadata['island_type'].values
    i = island_types[path_files.index(file)-1]
    
    all_g.append({
        "matrice": gg,
        "attributi": attributi_nodi,
        "island_type": i,
        "partizione_genere": partizione_indici,
        "nomi_per_genere": dizionario_nomi  # <-- Aggiunto qui!
    })

In [ ]:
all_g[0]["nomi_per_genere"]

In [ ]:
pri